In [ ]:
import os
os.chdir("..")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime
from astropy.time import Time
#from astropy.time import Time, TimeJD
from copy import deepcopy
from scipy.io import readsav
plt.style.use('default')
plt.style.use('bkj_style.mplstyle')
plt.rcParams['font.size'] = 11
from datetime import datetime, timedelta
import matplotlib.dates as mdates

_data = readsav("cak_koso.sav")
data = _data.data.T
time = _data.time
timeI = time.astype('long')

In [4]:
import pandas as pd
import numpy as np
from scipy.io import readsav

_data = readsav("cak_koso.sav")
data = _data.data.T
time = _data.time
timeI = time.astype('long')
#print(data.shape, timeI.shape)

_time = np.unique(time.astype('long'))
t0 = _time.min()
nx = _time.max()-t0+1
ny = 180
im = np.zeros((ny, nx))

for ii in _time:
    ind = np.where(timeI == ii)
    im_temp = data[:, ind[0]]
    im[:, ii-t0] = im_temp.mean(axis=1)



In [7]:
# full continuous integer-day axis
cols_full = np.arange(t0, t0 + nx, dtype=np.int64)
df_im = pd.DataFrame(im, index=np.linspace(-90, 90, ny), columns=cols_full)
df_im.index.name = 'Latitude (deg)'
df_im.columns.name = 'Day index (int)'


In [8]:
print(df_im.iloc[:10, :5])

Day index (int)  2416757  2416758  2416759  2416760  2416761
Latitude (deg)                                              
-90.000000           0.0      0.0      0.0      0.0      0.0
-88.994413           0.0      0.0      0.0      0.0      0.0
-87.988827           0.0      0.0      0.0      0.0      0.0
-86.983240           0.0      0.0      0.0      0.0      0.0
-85.977654           0.0      0.0      0.0      0.0      0.0
-84.972067           0.0      0.0      0.0      0.0      0.0
-83.966480           0.0      0.0      0.0      0.0      0.0
-82.960894           0.0      0.0      0.0      0.0      0.0
-81.955307           0.0      0.0      0.0      0.0      0.0
-80.949721           0.0      0.0      0.0      0.0      0.0


In [12]:
import h5py
import numpy as np
import pandas as pd
from astropy.time import Time

def ensure_native(a):
    """Return array with native (little-endian) byte order."""
    a = np.asarray(a)
    if a.dtype.kind in "uifb" and a.dtype.byteorder == ">":
        # big-endian -> swap to native
        return a.byteswap().newbyteorder()
    if a.dtype.byteorder == "=":  # already native
        return a
    if a.dtype.byteorder == "|":  # not byte-order dependent (e.g., bool, bytes)
        return a
    return a  # covers little-endian "<" as well

def build_lat_time_df(data, julday, latitude):
    data = ensure_native(data)
    lat  = ensure_native(latitude).ravel()
    jd   = ensure_native(julday).ravel()

    # detect which axis is latitude
    if data.shape[0] == lat.size:
        lat_axis = 0                     # (lat, time)
        time_len = data.shape[1]
    elif data.shape[1] == lat.size:
        lat_axis = 1                     # (time, lat)
        time_len = data.shape[0]
    else:
        raise ValueError(f"Latitude length {lat.size} matches neither axis of data {data.shape}")

    # align julian days with time axis
    jd_unique = np.unique(jd)
    if jd.size == time_len:
        jd_aligned = jd
    elif jd_unique.size == time_len:
        jd_aligned = jd_unique
    else:
        L = min(time_len, jd.size)
        jd_aligned = jd[:L]
        if lat_axis == 0:
            data = data[:, :L]
        else:
            data = data[:L, :]

    # orient to (time, lat)
    arr = data.T if lat_axis == 0 else data

    # convert JD -> datetime
    dates = Time(jd_aligned, format='jd').to_datetime()

    # build DataFrame
    df = pd.DataFrame(arr, columns=lat)
    df.insert(0, "Date", pd.to_datetime(dates))
    df.columns.name = "Latitude (deg)"
    return df

# ---- read and build ----
with h5py.File("KoSO_CaK_butterfly.h5", "r") as f:
    data1    = f["data1"][:]
    data2    = f["data2"][:]
    julday1  = f["julday1"][:]
    julday2  = f["julday2"][:]
    latitude = f["latitude"][:]

df1 = build_lat_time_df(data1, julday1, latitude)
df2 = build_lat_time_df(data2, julday2, latitude)

print("==== Data1 Table ====")
print(df1.head(), df1.shape)

print("\n==== Data2 Table ====")
print(df2.head(), df2.shape)


/home/dibya/anaconda3/envs/tfenv/lib/python3.9/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 32978 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/dibya/anaconda3/envs/tfenv/lib/python3.9/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "d2dtf" yielded 16696 of "dubious year (Note 5)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


==== Data1 Table ====
Latitude (deg)                       Date  -89.5  -88.5  -87.5  -86.5  -85.5  \
0              1904-10-04 06:30:00.000054    0.0    0.0    0.0    0.0    0.0   
1              1904-10-19 06:30:00.000054    0.0    0.0    0.0    0.0    0.0   
2              1904-10-19 06:30:00.000054    0.0    0.0    0.0    0.0    0.0   
3              1904-10-20 06:30:00.000054    0.0    0.0    0.0    0.0    0.0   
4              1904-10-26 18:31:00.000050    0.0    0.0    0.0    0.0    0.0   

Latitude (deg)  -84.5  -83.5  -82.5  -81.5  ...      80.5      81.5   82.5  \
0                 0.0    0.0    0.0    0.0  ...  0.000000  0.000000    0.0   
1                 0.0    0.0    0.0    0.0  ...  0.000000  0.000000    0.0   
2                 0.0    0.0    0.0    0.0  ...  0.000012  0.000001    0.0   
3                 0.0    0.0    0.0    0.0  ...  0.000000  0.000000    0.0   
4                 0.0    0.0    0.0    0.0  ...  0.000000  0.000000    0.0   

Latitude (deg)   83.5   84.5